# ??nh gi? model QTDB 1-k?nh theo protocol QTDB + NSTDB BW trong eval_new.txt

Notebook n?y b?m theo `eval_new.txt` ?? t?o b?ng b?o c?o gi?ng b?i g?c, d?ng checkpoint model 1-k?nh train ri?ng cho QTDB.

Protocol d? li?u:

- M?t m?u l? m?t beat ECG 1 k?nh, shape logic `(512, 1)`.
- Clean ECG l?y t? 14 test records c?a QT Database ???c li?t k? trong `eval_new.txt`.
- Noisy ECG ???c t?o b?ng c?ch c?ng baseline wander (`bw`) t? MIT-BIH Noise Stress Test Database.
- Ch?y c? `noise_type = 1` v? `noise_type = 2`, sau ?? g?p metric.
- ??nh gi? `1-shot`, `3-shot`, `5-shot`, `10-shot` b?ng SSD, MAD, PRD, Cosine similarity, SNR in, SNR out, SNR improvement.
- Xu?t b?ng `mean ? std` cho `ALL` v? t?ng kho?ng nhi?u.

Model d?ng trong notebook n?y l? `UNet1D` c?a b?n nh?ng ??i sang 1-k?nh: `in_channels=2`, `out_channels=1`. ??y l? ph??ng ?n ??i chi?u s?t b?i g?c nh?t.


## 1. Setup


In [ ]:
# Ch?y cell n?y tr?n Google Colab.
!pip -q install wfdb scipy pandas numpy matplotlib tqdm pyyaml scikit-learn


In [ ]:
from google.colab import drive

drive.mount('/content/drive')


In [ ]:
import gc
import math
import random
from pathlib import Path
from typing import Dict, Iterable, List, Optional, Tuple

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scipy.signal as signal
import torch
import torch.nn as nn
import torch.nn.functional as F
from tqdm.auto import tqdm
import wfdb

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', DEVICE)


## 2. Tham s? ch?y


In [ ]:
# Checkpoint model QTDB 1-k?nh train t? notebook Kaggle m?i.
CHECKPOINT_PATHS = {
    1: '/content/drive/MyDrive/phase1/checkpoints/qtdb_1ch_noise_type_1_best.pth',
    2: '/content/drive/MyDrive/phase1/checkpoints/qtdb_1ch_noise_type_2_best.pth',
}

# Test records ??ng theo eval_new.txt.
TEST_RECORDS = [
    'sel123', 'sel233', 'sel302', 'sel307', 'sel820', 'sel853',
    'sel16420', 'sel16795', 'sele0106', 'sele0121', 'sel32',
    'sel49', 'sel14046', 'sel15814',
]

# ?? None n?u mu?n d?ng ?? 14 records. Khi th? l?n ??u, gi?i h?n beat ?? tr?nh OOM.
MAX_RECORDS = None
MAX_BEATS_TOTAL = 512

# eval_new.txt ghi batch_size=50, nh?ng Colab c? th? OOM v?i diffusion nhi?u shot.
# D?ng 4-8 ?? ?n ??nh; metric kh?ng ??i v? m?t to?n h?c, ch? kh?c t?c ??.
SHOTS = [1, 3, 5, 10]
DDIM_STEPS = 50
DDIM_ETA = 0.0
BATCH_SIZE = 4

# Protocol d? li?u b?i g?c: beat 1 k?nh, d?i 512, fs=360 Hz.
TARGET_FS = 360
BEAT_LENGTH = 512
EVAL_CHANNELS = 1
NOISE_LEVEL_RANGE = (0.2, 1.99)
NOISE_BINS = [
    ('0.2-0.6', 0.2, 0.6),
    ('0.6-1.0', 0.6, 1.0),
    ('1.0-1.5', 1.0, 1.5),
    ('1.5-2.0', 1.5, 2.0),
]

# Model 1-k?nh cho Ph??ng ?n A.
MODEL_CHANNELS = 1
BASE_FEATS = 80
EMB_DIM = 128
MODEL_ARCH = 'unet1d_qtdb_1ch'


## 3. ??nh ngh?a model UNet1D c?a b?n v? diffusion sampler


In [ ]:
class HNFBlockUNet(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_sizes=(3, 5, 9, 15)):
        super().__init__()
        self.multi_convs = nn.ModuleList([
            nn.Conv1d(in_channels, out_channels // len(kernel_sizes), k, padding=k // 2, padding_mode='reflect')
            for k in kernel_sizes
        ])
        self.agg_conv = nn.Conv1d(out_channels, out_channels, 1)
        self.half_inst_norm = nn.InstanceNorm1d(out_channels // 2)
        self.act = nn.ReLU(inplace=True)
        self.residual = nn.Conv1d(in_channels, out_channels, 1) if in_channels != out_channels else nn.Identity()

    def forward(self, x):
        multi_out = [conv(x) for conv in self.multi_convs]
        out = torch.cat(multi_out, dim=1)
        out = self.agg_conv(out)
        half = out.shape[1] // 2
        out_norm = self.half_inst_norm(out[:, :half, :])
        out = torch.cat([out_norm, out[:, half:, :]], dim=1)
        out = self.act(out)
        return out + self.residual(x)


class BridgeBlockUNet(nn.Module):
    def __init__(self, features, emb_dim=128):
        super().__init__()
        self.emb_dim = emb_dim
        self.film = nn.Sequential(
            nn.Linear(emb_dim, features * 2),
            nn.SiLU(),
        )

    def sinusoidal_embedding(self, x):
        if x.dim() == 0:
            x = x.unsqueeze(0)
        device = x.device
        half_dim = self.emb_dim // 2
        emb = math.log(10000) / (half_dim - 1)
        emb = torch.exp(torch.arange(half_dim, device=device) * -emb)
        emb = x.unsqueeze(-1) * emb.unsqueeze(0)
        return torch.cat([torch.sin(emb), torch.cos(emb)], dim=-1)

    def forward(self, x, alpha_bar):
        alpha_bar = alpha_bar.view(-1)
        emb = self.sinusoidal_embedding(alpha_bar)
        scale_shift = self.film(emb)
        scale, shift = scale_shift.chunk(2, dim=1)
        return x * (1 + scale.unsqueeze(-1)) + shift.unsqueeze(-1)


class SelfAttention1D(nn.Module):
    def __init__(self, channels, num_heads=4):
        super().__init__()
        self.num_heads = num_heads
        self.head_dim = channels // num_heads
        assert self.head_dim * num_heads == channels
        self.qkv = nn.Conv1d(channels, channels * 3, kernel_size=1)
        self.proj = nn.Conv1d(channels, channels, kernel_size=1)

    def forward(self, x):
        batch, channels, length = x.shape
        qkv = self.qkv(x).reshape(batch, 3, self.num_heads, self.head_dim, length)
        q, k, v = qkv[:, 0], qkv[:, 1], qkv[:, 2]
        attn = torch.matmul(q.transpose(-2, -1), k) / (self.head_dim ** 0.5)
        attn = torch.softmax(attn, dim=-1)
        out = torch.matmul(attn, v.transpose(-2, -1)).transpose(-2, -1)
        out = out.reshape(batch, channels, length)
        return self.proj(out)


class UNet1D(nn.Module):
    def __init__(self, in_channels=24, base_channels=64, emb_dim=128, out_channels=12):
        super().__init__()
        self.enc1 = HNFBlockUNet(in_channels, base_channels)
        self.bridge1 = BridgeBlockUNet(base_channels, emb_dim)
        self.down1 = nn.Conv1d(base_channels, base_channels * 2, kernel_size=4, stride=2, padding=1)

        self.enc2 = HNFBlockUNet(base_channels * 2, base_channels * 2)
        self.bridge2 = BridgeBlockUNet(base_channels * 2, emb_dim)
        self.down2 = nn.Conv1d(base_channels * 2, base_channels * 4, kernel_size=4, stride=2, padding=1)

        self.enc3 = HNFBlockUNet(base_channels * 4, base_channels * 4)
        self.bridge3 = BridgeBlockUNet(base_channels * 4, emb_dim)
        self.down3 = nn.Conv1d(base_channels * 4, base_channels * 8, kernel_size=4, stride=2, padding=1)

        self.enc4 = HNFBlockUNet(base_channels * 8, base_channels * 8)
        self.bridge4 = BridgeBlockUNet(base_channels * 8, emb_dim)
        self.attn = SelfAttention1D(base_channels * 8)

        self.up4 = nn.ConvTranspose1d(base_channels * 8, base_channels * 4, kernel_size=4, stride=2, padding=1)
        self.dec4 = HNFBlockUNet(base_channels * 8, base_channels * 4)

        self.up3 = nn.ConvTranspose1d(base_channels * 4, base_channels * 2, kernel_size=4, stride=2, padding=1)
        self.dec3 = HNFBlockUNet(base_channels * 4, base_channels * 2)

        self.up2 = nn.ConvTranspose1d(base_channels * 2, base_channels, kernel_size=4, stride=2, padding=1)
        self.dec2 = HNFBlockUNet(base_channels * 2, base_channels)

        self.final = nn.Conv1d(base_channels, out_channels, kernel_size=1)

    def forward(self, x, cond, noise_scale):
        inp = torch.cat([x, cond], dim=1)

        e1 = self.bridge1(self.enc1(inp), noise_scale)
        d1 = self.down1(e1)

        e2 = self.bridge2(self.enc2(d1), noise_scale)
        d2 = self.down2(e2)

        e3 = self.bridge3(self.enc3(d2), noise_scale)
        d3 = self.down3(e3)

        e4 = self.bridge4(self.enc4(d3), noise_scale)
        e4 = self.attn(e4)

        u4 = self.up4(e4)
        d4 = self.dec4(torch.cat([u4, e3], dim=1))

        u3 = self.up3(d4)
        d3 = self.dec3(torch.cat([u3, e2], dim=1))

        u2 = self.up2(d3)
        d2 = self.dec2(torch.cat([u2, e1], dim=1))

        return self.final(d2)


In [ ]:
def _extract(buffer, t, shape):
    value = buffer.gather(0, t)
    return value.reshape((shape[0],) + (1,) * (len(shape) - 1))


class DDPM(nn.Module):
    def __init__(self, base_model, num_steps=50, beta_start=1e-4, beta_end=0.5, schedule='quad'):
        super().__init__()
        self.model = base_model
        self.num_steps = num_steps
        betas = self.make_beta_schedule(schedule, num_steps, beta_start, beta_end)
        alphas = 1.0 - betas
        alphas_cumprod = torch.cumprod(alphas, dim=0)
        alphas_cumprod_prev = torch.cat([torch.ones(1), alphas_cumprod[:-1]])

        self.register_buffer('betas', betas.float())
        self.register_buffer('alphas_cumprod', alphas_cumprod.float())
        self.register_buffer('alphas_cumprod_prev', alphas_cumprod_prev.float())
        self.register_buffer('sqrt_alphas_cumprod', torch.sqrt(alphas_cumprod).float())
        self.register_buffer('sqrt_one_minus_alphas_cumprod', torch.sqrt(1.0 - alphas_cumprod).float())
        self.register_buffer('sqrt_recip_alphas_cumprod', torch.sqrt(1.0 / alphas_cumprod).float())
        self.register_buffer('sqrt_recipm1_alphas_cumprod', torch.sqrt(1.0 / alphas_cumprod - 1).float())

        posterior_variance = betas * (1.0 - alphas_cumprod_prev) / (1.0 - alphas_cumprod)
        self.register_buffer('posterior_variance', posterior_variance.float())
        self.register_buffer('posterior_log_variance_clipped', torch.log(torch.clamp(posterior_variance, min=1e-20)).float())
        self.register_buffer('posterior_mean_coef1', (betas * torch.sqrt(alphas_cumprod_prev) / (1.0 - alphas_cumprod)).float())
        self.register_buffer('posterior_mean_coef2', ((1.0 - alphas_cumprod_prev) * torch.sqrt(alphas) / (1.0 - alphas_cumprod)).float())

    @staticmethod
    def make_beta_schedule(schedule, n_timesteps, start, end):
        if schedule == 'linear':
            return torch.linspace(start, end, n_timesteps)
        if schedule == 'quad':
            return torch.linspace(start ** 0.5, end ** 0.5, n_timesteps) ** 2
        if schedule == 'sigmoid':
            values = torch.linspace(-6, 6, n_timesteps)
            return torch.sigmoid(values) * (end - start) + start
        raise ValueError(f'Unsupported beta schedule: {schedule}')

    def predict_start_from_noise(self, x_t, t, noise):
        return _extract(self.sqrt_recip_alphas_cumprod, t, x_t.shape) * x_t - _extract(self.sqrt_recipm1_alphas_cumprod, t, x_t.shape) * noise

    @torch.no_grad()
    def ddim_sample_loop(self, condition, ddim_timesteps=50, ddim_eta=0.0, num_shots=1):
        device = condition.device
        total_steps = self.num_steps
        sample_steps = min(ddim_timesteps, total_steps)
        tau = [int(np.floor((total_steps / (sample_steps ** 2)) * (i ** 2))) for i in range(sample_steps + 1)]
        alphas_with_one = torch.cat([torch.ones(1, device=device, dtype=self.alphas_cumprod.dtype), self.alphas_cumprod])

        out_accum = torch.zeros_like(condition)
        for _ in range(num_shots):
            x = torch.randn_like(condition)
            for i in reversed(range(1, sample_steps + 1)):
                t_value = tau[i]
                t_prev = tau[i - 1]
                noise_level = torch.full((x.shape[0], 1), float(torch.sqrt(alphas_with_one[t_value])), device=device)
                eps = self.model(x, condition, noise_level)
                a_t = alphas_with_one[t_value]
                a_prev = alphas_with_one[t_prev]
                x0_pred = (x - torch.sqrt(1.0 - a_t) * eps) / torch.sqrt(a_t)
                if t_prev == 0:
                    sigma_t = 0.0
                else:
                    sigma_t = ddim_eta * torch.sqrt((1.0 - a_prev) / (1.0 - a_t)) * torch.sqrt(torch.clamp(1.0 - a_t / a_prev, min=0.0))
                direction = torch.sqrt(torch.clamp(1.0 - a_prev - sigma_t ** 2, min=0.0)) * eps
                noise = sigma_t * torch.randn_like(x) if (ddim_eta > 0 and t_prev > 0) else 0.0
                x = torch.sqrt(a_prev) * x0_pred + direction + noise
            out_accum += x
        return out_accum / num_shots


def build_model(checkpoint_path):
    if MODEL_ARCH != 'unet1d_qtdb_1ch':
        raise ValueError("Notebook n?y ?ang c?u h?nh cho model QTDB 1-k?nh: MODEL_ARCH='unet1d_qtdb_1ch'")

    base_model = UNet1D(
        in_channels=MODEL_CHANNELS * 2,
        base_channels=BASE_FEATS,
        emb_dim=EMB_DIM,
        out_channels=MODEL_CHANNELS,
    ).to(DEVICE)
    model = DDPM(base_model, num_steps=50, beta_start=1e-4, beta_end=0.5, schedule='quad').to(DEVICE)

    checkpoint = torch.load(checkpoint_path, map_location=DEVICE)
    if isinstance(checkpoint, dict) and 'state_dict' in checkpoint:
        checkpoint = checkpoint['state_dict']
    checkpoint = {k.replace('module.', ''): v for k, v in checkpoint.items()}

    missing, unexpected = model.load_state_dict(checkpoint, strict=False)
    print(f'Loaded checkpoint: {checkpoint_path}')
    print(f'Missing keys: {len(missing)} | Unexpected keys: {len(unexpected)}')
    if unexpected[:5]:
        print('Unexpected key examples:', unexpected[:5])
    if missing[:5]:
        print('Missing key examples:', missing[:5])
    if missing or unexpected:
        raise RuntimeError('Checkpoint kh?ng kh?p ho?n to?n v?i UNet1D 1-k?nh. H?y d?ng checkpoint train b?ng notebook Kaggle QTDB 1ch.')
    model.eval()
    return model


## 4. T?i v? chu?n b? ECG s?ch t? QTDB


In [ ]:
BEAT_SYMBOLS = {
    'N', 'L', 'R', 'B', 'A', 'a', 'J', 'S', 'V', 'r', 'F', 'e', 'j', 'n', 'E', '/', 'f', 'Q', '?'
}


def read_qtdb_annotation(record_name):
    last_error = None
    for extension in ['atr', 'pu', 'q1c', 'q2c']:
        try:
            ann = wfdb.rdann(record_name, extension, pn_dir='qtdb/1.0.0')
            return ann, extension
        except Exception as exc:
            last_error = exc
    raise RuntimeError(f'Cannot read a usable annotation for QTDB record {record_name}: {last_error}')


def annotation_centers(annotation):
    symbols = np.asarray(annotation.symbol)
    samples = np.asarray(annotation.sample)
    beat_mask = np.asarray([symbol in BEAT_SYMBOLS for symbol in symbols])
    if beat_mask.any():
        return samples[beat_mask]
    return samples


def resample_to_target_fs(ecg, source_fs, target_fs=360):
    if int(source_fs) == int(target_fs):
        return ecg.astype(np.float32)
    target_len = int(round(ecg.shape[0] * target_fs / source_fs))
    return signal.resample(ecg, target_len, axis=0).astype(np.float32)


def select_eval_channel(ecg, channel_index=0):
    if ecg.ndim == 1:
        ecg = ecg[:, None]
    return ecg[:, channel_index:channel_index + 1].astype(np.float32)


def baseline_correct(segment):
    return segment - np.median(segment, axis=0, keepdims=True)


def normalize_segment(segment, eps=1e-8):
    segment = baseline_correct(segment)
    scale = np.max(np.abs(segment), axis=0, keepdims=True)
    return (segment / (scale + eps)).astype(np.float32)


def crop_beat(ecg, center, length=512):
    half = length // 2
    start = int(center) - half
    end = start + length
    if start < 0 or end > len(ecg):
        return None
    return ecg[start:end]


def load_qtdb_clean_beats(test_records, max_records=None, max_beats_total=None):
    records = list(test_records)
    if max_records is not None:
        records = records[:max_records]

    beats = []
    meta_rows = []
    for record_name in tqdm(records, desc='QTDB test records'):
        try:
            record = wfdb.rdrecord(record_name, pn_dir='qtdb/1.0.0')
            ann, ann_extension = read_qtdb_annotation(record_name)
            ecg = resample_to_target_fs(record.p_signal, record.fs, TARGET_FS)
            ecg = select_eval_channel(ecg, channel_index=0)
            ratio = TARGET_FS / float(record.fs)
            centers = np.round(annotation_centers(ann) * ratio).astype(int)

            kept_for_record = 0
            for center in centers:
                beat = crop_beat(ecg, center, BEAT_LENGTH)
                if beat is None:
                    continue
                beat = normalize_segment(beat)
                if np.all(np.isfinite(beat)) and beat.shape == (BEAT_LENGTH, EVAL_CHANNELS):
                    beats.append(beat)
                    kept_for_record += 1
                    meta_rows.append({'record': record_name, 'annotation': ann_extension, 'center_sample': int(center)})
                if max_beats_total is not None and len(beats) >= max_beats_total:
                    print(f'Reached MAX_BEATS_TOTAL={max_beats_total}')
                    return np.stack(beats).astype(np.float32), pd.DataFrame(meta_rows)
            print(f'{record_name}: kept {kept_for_record} beats')
            del record, ann, ecg, centers
            gc.collect()
        except Exception as exc:
            print(f'Skip {record_name}: {exc}')
            gc.collect()

    if not beats:
        raise RuntimeError('No QTDB beats were loaded. Check Colab internet access and the QTDB record names.')
    return np.stack(beats).astype(np.float32), pd.DataFrame(meta_rows)


clean_beats, beat_meta = load_qtdb_clean_beats(TEST_RECORDS, MAX_RECORDS, MAX_BEATS_TOTAL)
print('clean_beats shape:', clean_beats.shape, '(expected: N, 512, 1)')
display(beat_meta.groupby('record').size().rename('beats').reset_index())
display(beat_meta.head())


## 5. T?i BW noise v? t?o test set cho noise_type 1/2


In [ ]:
def load_bw_noise():
    record = wfdb.rdrecord('bw', pn_dir='nstdb/1.0.0')
    noise = resample_to_target_fs(record.p_signal, record.fs, TARGET_FS)
    if noise.shape[1] < 2:
        noise = np.tile(noise, (1, 2))
    return noise.astype(np.float32)


def split_noise_for_type(noise, noise_type):
    middle = len(noise) // 2
    if noise_type == 1:
        train_noise = noise[:middle, 0]
        test_noise = noise[middle:, 1]
    elif noise_type == 2:
        train_noise = noise[:middle, 1]
        test_noise = noise[middle:, 0]
    else:
        raise ValueError('noise_type must be 1 or 2')
    return train_noise, test_noise


def amplitude_range(x, eps=1e-8):
    return float(np.max(x) - np.min(x) + eps)


def make_noisy_batch(clean_batch, noise, rng):
    noisy = np.zeros_like(clean_batch, dtype=np.float32)
    levels = np.zeros(clean_batch.shape[0], dtype=np.float32)
    starts = np.zeros(clean_batch.shape[0], dtype=np.int64)

    if len(noise) <= BEAT_LENGTH:
        raise ValueError('BW noise signal is shorter than one beat window')

    for i, beat in enumerate(clean_batch):
        start = int(rng.integers(0, len(noise) - BEAT_LENGTH))
        noise_patch = noise[start:start + BEAT_LENGTH].astype(np.float32)
        noise_patch = noise_patch - np.median(noise_patch)
        noise_multi = np.repeat(noise_patch[:, None], beat.shape[1], axis=1)
        ase = amplitude_range(noise_patch) / amplitude_range(beat)
        level = float(rng.uniform(NOISE_LEVEL_RANGE[0], NOISE_LEVEL_RANGE[1]))
        alpha = level / ase
        noisy[i] = beat + alpha * noise_multi
        levels[i] = level
        starts[i] = start
    return noisy, levels, starts


bw_noise = load_bw_noise()
test_noise_by_type = {}
for noise_type in [1, 2]:
    _, test_noise_by_type[noise_type] = split_noise_for_type(bw_noise, noise_type)
    print(f'noise_type={noise_type}: test BW samples={len(test_noise_by_type[noise_type])}')

# Gi?i ph?ng b?n noise 2 k?nh sau khi t?ch ?? gi?m RAM.
del bw_noise


## 6. Ch?y denoising


In [ ]:
models_by_noise_type = {}
for noise_type, checkpoint_path in CHECKPOINT_PATHS.items():
    checkpoint = Path(checkpoint_path)
    if not checkpoint.is_file():
        raise FileNotFoundError(
            f'Kh?ng t?m th?y checkpoint cho noise_type={noise_type}: {checkpoint_path}\n'
            'H?y upload checkpoint QTDB 1-k?nh t? Kaggle l?n Drive r?i s?a CHECKPOINT_PATHS.'
        )
    models_by_noise_type[noise_type] = build_model(str(checkpoint))


In [ ]:
def to_model_tensor(batch_np):
    # Protocol eval_new.txt: numpy (B, 512, 1) -> torch (B, 1, 512)
    if batch_np.shape[-1] != MODEL_CHANNELS:
        raise ValueError(f'Expected {MODEL_CHANNELS} channel(s), got shape {batch_np.shape}')
    return torch.from_numpy(batch_np).float().permute(0, 2, 1).to(DEVICE)


@torch.no_grad()
def denoise_batch(model, noisy_np, shots):
    batch = to_model_tensor(noisy_np)
    denoised = model.ddim_sample_loop(batch, ddim_timesteps=DDIM_STEPS, ddim_eta=DDIM_ETA, num_shots=shots)
    out = denoised.detach().cpu().permute(0, 2, 1).numpy().astype(np.float32)
    del batch, denoised
    if DEVICE == 'cuda':
        torch.cuda.empty_cache()
    return out


print('Eval data and model are both one-channel: (N, 512, 1) -> (N, 1, 512).')


## 7. T?nh metric theo batch v? t?o b?ng k?t qu?


In [ ]:
def flatten_signals(x):
    return x.reshape(x.shape[0], -1)


def metric_rows(clean, noisy, denoised, noise_level, shot, noise_type, global_start_index):
    clean_f = flatten_signals(clean)
    noisy_f = flatten_signals(noisy)
    denoised_f = flatten_signals(denoised)

    signal_power = np.sum(clean_f ** 2, axis=1)
    noisy_error_power = np.sum((noisy_f - clean_f) ** 2, axis=1)
    denoised_error_power = np.sum((denoised_f - clean_f) ** 2, axis=1)

    ssd = denoised_error_power
    mad = np.max(np.abs(denoised_f - clean_f), axis=1)
    prd = np.sqrt(denoised_error_power / (np.sum((denoised_f - np.mean(clean_f, axis=1, keepdims=True)) ** 2, axis=1) + 1e-8)) * 100
    cosine = np.sum(clean_f * denoised_f, axis=1) / ((np.linalg.norm(clean_f, axis=1) * np.linalg.norm(denoised_f, axis=1)) + 1e-8)
    snr_in = 10 * np.log10((signal_power + 1e-8) / (noisy_error_power + 1e-8))
    snr_out = 10 * np.log10((signal_power + 1e-8) / (denoised_error_power + 1e-8))

    return pd.DataFrame({
        'sample_index': np.arange(global_start_index, global_start_index + len(clean)),
        'shot': shot,
        'noise_type': noise_type,
        'noise_level': noise_level,
        'SSD': ssd,
        'MAD': mad,
        'PRD': prd,
        'Cosine similarity': cosine,
        'SNR in': snr_in,
        'SNR out': snr_out,
        'SNR improvement': snr_out - snr_in,
    })


In [ ]:
METRIC_COLUMNS = ['SSD', 'MAD', 'PRD', 'Cosine similarity', 'SNR in', 'SNR out', 'SNR improvement']


def mean_std_text(series):
    return f'{series.mean():.4f} ? {series.std(ddof=1):.4f}'


def summarize_group(df, label):
    row = {'Group': label, 'N': len(df)}
    for metric in METRIC_COLUMNS:
        row[metric] = mean_std_text(df[metric]) if len(df) > 1 else f'{df[metric].mean():.4f} ? nan'
    return row


def build_summary(metrics_df):
    summary_rows = []
    for shot, shot_df in metrics_df.groupby('shot', sort=True):
        summary_rows.append({'Shot': shot, **summarize_group(shot_df, 'ALL')})
        for label, low, high in NOISE_BINS:
            bin_df = shot_df[(shot_df['noise_level'] > low) & (shot_df['noise_level'] < high)]
            if len(bin_df) == 0:
                continue
            summary_rows.append({'Shot': shot, **summarize_group(bin_df, label)})
    return pd.DataFrame(summary_rows)


metric_parts = []
sample_plot_cache = None

for noise_type in [1, 2]:
    model = models_by_noise_type[noise_type]
    rng = np.random.default_rng(SEED + noise_type)
    test_noise = test_noise_by_type[noise_type]

    for batch_start in tqdm(range(0, len(clean_beats), BATCH_SIZE), desc=f'noise_type={noise_type}'):
        clean_batch = clean_beats[batch_start:batch_start + BATCH_SIZE]
        noisy_batch, levels_batch, _ = make_noisy_batch(clean_batch, test_noise, rng)

        for shot in SHOTS:
            denoised_batch = denoise_batch(model, noisy_batch, shots=shot)
            metric_parts.append(metric_rows(
                clean_batch,
                noisy_batch,
                denoised_batch,
                levels_batch,
                shot,
                noise_type,
                batch_start,
            ))
            if sample_plot_cache is None:
                sample_plot_cache = {
                    'noise_type': noise_type,
                    'shot': shot,
                    'clean': clean_batch[0].copy(),
                    'noisy': noisy_batch[0].copy(),
                    'denoised': denoised_batch[0].copy(),
                }
            del denoised_batch

        del clean_batch, noisy_batch, levels_batch

metrics_df = pd.concat(metric_parts, ignore_index=True)
summary_df = build_summary(metrics_df)

display(metrics_df.head())
print('metric rows:', len(metrics_df))
display(summary_df)


## 8. Sanity diagnostics

C?c ki?m tra n?y gi?p ph?n bi?t l?i d? li?u/protocol v?i l?i model. N?u `Noisy vs Clean` c? cosine cao nh?ng `Denoised vs Clean` r?t th?p, d? li?u noisy kh?ng ph?i nguy?n nh?n ch?nh; v?n ?? n?m ? model/protocol inference.


In [ ]:
def cosine_np(a, b):
    af = flatten_signals(a)
    bf = flatten_signals(b)
    return np.sum(af * bf, axis=1) / ((np.linalg.norm(af, axis=1) * np.linalg.norm(bf, axis=1)) + 1e-8)


# Recreate one small deterministic diagnostic batch.
diag_noise_type = 1
diag_rng = np.random.default_rng(SEED + diag_noise_type)
diag_clean = clean_beats[:min(BATCH_SIZE, len(clean_beats))]
diag_noisy, diag_levels, _ = make_noisy_batch(diag_clean, test_noise_by_type[diag_noise_type], diag_rng)
diag_denoised = denoise_batch(model, diag_noisy, shots=1)

sanity = pd.DataFrame({
    'check': ['Noisy vs Clean', 'Denoised vs Clean', '-Denoised vs Clean'],
    'mean_cosine': [
        cosine_np(diag_clean, diag_noisy).mean(),
        cosine_np(diag_clean, diag_denoised).mean(),
        cosine_np(diag_clean, -diag_denoised).mean(),
    ],
    'clean_std': [diag_clean.std(), diag_clean.std(), diag_clean.std()],
    'other_std': [diag_noisy.std(), diag_denoised.std(), (-diag_denoised).std()],
    'clean_min': [diag_clean.min(), diag_clean.min(), diag_clean.min()],
    'clean_max': [diag_clean.max(), diag_clean.max(), diag_clean.max()],
    'other_min': [diag_noisy.min(), diag_denoised.min(), (-diag_denoised).min()],
    'other_max': [diag_noisy.max(), diag_denoised.max(), (-diag_denoised).max()],
})
display(sanity)

print('Diagnostic interpretation:')
print('- If Noisy vs Clean cosine is much higher than Denoised vs Clean, the model output is damaging the signal.')
print('- If -Denoised vs Clean is high, output polarity is flipped.')
print('- If denoised std/min/max is far from clean, there is a scale/domain mismatch.')


## 9. L?u k?t qu? v? v? ki?m tra nhanh


In [ ]:
output_dir = Path('/content/eval_outputs')
output_dir.mkdir(parents=True, exist_ok=True)

metrics_path = output_dir / 'eval_metrics_per_beat.csv'
summary_path = output_dir / 'eval_summary_mean_std.csv'
metrics_df.to_csv(metrics_path, index=False)
summary_df.to_csv(summary_path, index=False)

print('Saved:', metrics_path)
print('Saved:', summary_path)

# Copy sang Drive n?u mu?n gi? sau khi runtime Colab t?t.
drive_output_dir = Path('/content/drive/MyDrive/phase1/eval_outputs')
drive_output_dir.mkdir(parents=True, exist_ok=True)
metrics_df.to_csv(drive_output_dir / metrics_path.name, index=False)
summary_df.to_csv(drive_output_dir / summary_path.name, index=False)
print('Copied to:', drive_output_dir)


In [ ]:
# V? m?t beat m?u ?? cache trong l?c ch?y streaming.
if sample_plot_cache is None:
    raise RuntimeError('No sample plot cache found. Run the streaming evaluation cell first.')

lead_names = ['I', 'II', 'III', 'aVR', 'aVL', 'aVF', 'V1', 'V2', 'V3', 'V4', 'V5', 'V6']
clean = sample_plot_cache['clean']
noisy = sample_plot_cache['noisy']
denoised = sample_plot_cache['denoised']

fig, axes = plt.subplots(6, 2, figsize=(16, 18), sharex=True)
axes = axes.ravel()
for lead_idx, ax in enumerate(axes):
    ax.plot(clean[:, lead_idx], color='black', linewidth=1.5, label='Clean')
    ax.plot(noisy[:, lead_idx], color='gray', alpha=0.45, label='Noisy')
    ax.plot(denoised[:, lead_idx], color='crimson', linewidth=1.0, label='Denoised')
    ax.set_title(lead_names[lead_idx])
    ax.grid(alpha=0.25)
    if lead_idx == 0:
        ax.legend(loc='upper right')
plt.suptitle(
    f"Sanity check: noise_type={sample_plot_cache['noise_type']}, {sample_plot_cache['shot']}-shot",
    y=1.01,
)
plt.tight_layout()
plot_path = output_dir / 'sample_denoising_plot.png'
plt.savefig(plot_path, dpi=180, bbox_inches='tight')
plt.show()
print('Saved:', plot_path)


## 10. Di?n gi?i k?t qu?

Sau khi ch?y xong, ??c b?ng `summary_df`:

- `SSD`, `MAD`, `PRD` c?ng nh? c?ng t?t.
- `Cosine similarity` c?ng g?n 1 c?ng t?t.
- `SNR improvement` c?ng l?n c?ng t?t.
- H?ng `ALL` l? to?n b? beat c?a c? `noise_type=1` v? `noise_type=2`.
- C?c h?ng `0.2-0.6`, `0.6-1.0`, `1.0-1.5`, `1.5-2.0` l? k?t qu? theo m?c nhi?u ban ??u `r`.

Notebook n?y d?nh cho Ph??ng ?n A: train/eval model 1-k?nh tr?n QTDB ?? ??i chi?u s?t b?i g?c. Kh?ng d?ng adapter l?p 12 k?nh n?a.
